# Data Quality Dashboard

`src/validate_dataset.py` dosyasının kabul kriterlerini görselleştirir. Terminale
metin olarak basılan ölçümleri tek ekranda toplar: hacim, label dağılımı, `normal`
sınıfındaki runtime kapsamı (Problem B), senaryo × feature ayrılabilirliği, window boyutu
karşılaştırması ve archive kıyası.

**Bu notebook kimin için?** Senin için. Her `make features` sonrası bir kez çalıştırıp
"veri seti FAZ 4'e hazır mı" sorusunun cevabını alırsın. Danışana gösterim için
`advisor-demo.ipynb`, ham veriyi incelemek için `parquet-explorer.ipynb`.

## İki kural

**1. Ölçüm mantığı notebook'ta yazılmaz.** `normal_runtime_coverage`, `separability`,
`best_discriminator`, `is_degenerate`, `check_targets` ve `TARGETS`, hepsi
`src/validate_dataset.py` dosyasından import ediliyor; feature şeması
`src/features/extract_features.py` dosyasından geliyor. Görünüm katmanı ise `style.py`
ve `charts.py` içinde: ortak palette ve birden çok notebook'un paylaştığı grafikler.

**2. Bu notebook'ta elle yazılmış ölçüm yok.** Her sayı çalıştırma anında
`features.parquet` dosyasından hesaplanıyor. Kanonik sayılar `roadmap/00-durum.md`
dosyasında yaşıyor; buraya kopyalanmıyor, çünkü kopyalanan sayı eskiyor.

> **Sözlük notu:** Teknik terimler İngilizce (dashboard, feature, label, window,
> window, baseline, fusion), açıklamalar Türkçe. `make validate` tablosunun
> satır adları `validate_dataset.py` içinden geldiği için Türkçe kalıyor; ikisi aynı
> ölçümün iki yüzü ve tek kaynak tek kalmalı.

**Nasıl çalıştırılır:** `make nb` (JupyterLab) veya `make nb-exec` (başsız doğrulama).

## 1 · Kurulum

Repo kökünü bulup `src/` klasörünü import yoluna ekliyor, ölçüm modülleriyle ortak
görünüm katmanını yüklüyor.

In [ ]:
import sys
from pathlib import Path

# `style.py` bu notebook'un yanında (notebooks/). Kernel cwd'si normalde notebook'un
# dizinidir (JupyterLab ve nbconvert ikisi de böyle); JupyterLab'in kökü repo kökü
# verilmişse diye tek adım aşağısı da denenir.
HERE = Path.cwd().resolve()
if not (HERE / "style.py").is_file():
    HERE = HERE / "notebooks"
sys.path.insert(0, str(HERE))

import style

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display

import charts  # birden çok notebook'un paylaştığı grafikler

REPO = style.bootstrap()      # repo kökü + src/ import yolunda
style.apply_theme()

# Ölçüm mantığının TEK tek kaynağı — burada hiçbir ölçüm yazılmaz:
import validate_dataset as vd
from features.extract_features import ALL_COLS, NET_COLS, RT_COLS

print(f"repo kökü      : {REPO}")
print(f"feature şeması : {len(ALL_COLS)} feature ({len(NET_COLS)} network + {len(RT_COLS)} runtime)")
print(f"TARGETS        : {vd.TARGETS}")
print(f"ayrılabilirlik eşiği: {vd.DISCRIMINATOR_THRESHOLD} normal σ")

## 2 · Veri kaynağı ve tazelik

Bir dashboard'un en tehlikeli hâli, eski veriyle doğru görünmesidir. Bu hücre
`features.parquet` dosyasının ham toplamadan sonra üretilip üretilmediğini kontrol
ediyor: `data/raw/` altındaki part dosyaları `features.parquet` dosyasından yeniyse
dashboard eski veriyi gösteriyordur.

In [ ]:
FEATURES = REPO / "src/data/features/features.parquet"
RAW_DIR = REPO / "src/data/raw"
WINDOW = "10s"      # `make features WINDOW=10s` ile üretilen window


def stamp(path):
    return pd.Timestamp(path.stat().st_mtime, unit="s", tz="UTC")


def freshness(features_path, raw_dir):
    lines = [f"features.parquet : {stamp(features_path):%Y-%m-%d %H:%M} UTC"]
    parts = sorted(raw_dir.glob("*.part*.parquet"))
    merged = sorted(raw_dir.glob("*.parquet"))
    stale = False
    if parts:
        newest = max(parts, key=lambda p: p.stat().st_mtime)
        lines.append(f"ham part dosyası : {len(parts)} adet · en yenisi {stamp(newest):%Y-%m-%d %H:%M} UTC")
        stale = stamp(newest) > stamp(features_path)
    else:
        lines.append("ham part dosyası : yok")
    if not merged:
        lines.append("merged parquet   : YOK — collection koşusu sürüyor olabilir "
                     "(merged dosya koşu bitince yazılır)")
    if stale:
        lines += ["", "HAM VERİ DAHA YENİ — BU DASHBOARD ESKİMİŞ.",
                  "    Koşu sürüyor ya da bittikten sonra feature üretilmedi.",
                  "    Önce:  make features WINDOW=10s"]
    return "\n".join(lines)


df = pd.read_parquet(FEATURES)
print(freshness(FEATURES, RAW_DIR))
print(f"\nrows={len(df)} · columns={len(df.columns)} · labels={df.label.nunique()}")

In [ ]:
summary = vd.summarize(df)
coverage = vd.normal_runtime_coverage(df)
print(f"summary: {summary}")

## 3 · Hacim ve label dağılımı

**Neden:** senaryo başına birkaç satırla model eğitilemez, doğrulanamaz, ablation
yapılamaz.

Eşikler `vd.TARGETS` içinde tanımlı ve grafik onları kendisi basıyor. `normal` ile
senaryoların eşikleri farklı olduğu için tek bir yatay çizgi ikisini temsil edemez; her
çubuğun kendi hedefi dikey bir işaretle gösteriliyor.

In [ ]:
fig, overview = charts.label_distribution(df, WINDOW)
overview["status"] = [style.status_text(w >= t)
                      for w, t in zip(overview["window sayısı"], overview["hedef"])]
plt.show()
overview

## 4 · Problem B — `normal` sınıfında runtime kapsamı

**Neden bu dashboard'un merkezinde:** `normal` sınıfının runtime feature'ları tamamen
sıfıra yakınsa, `exec_count > 0` tek başına normali neredeyse kusursuz ayırır. Model
"normal runtime davranışı" diye öğrenilecek bir şey bulamaz; yalnızca "olay varsa
saldırı" kuralını öğrenir. Bu, tezin feature seviyesi füzyon iddiasını boşa çıkarır,
çünkü runtime katmanı sıfıra yakın bilgi taşır.

Ölçüt: `normal` satırların en az bir runtime feature'ı sıfırdan farklı olma oranı
(`vd.normal_runtime_coverage`); kabul eşiği `vd.TARGETS` içinde tanımlı.

In [ ]:
fig, coverage = charts.coverage_panel(df)
plt.show()

## 5 · Senaryo × feature ayrılabilirliği

**Neden:** kabul kriterleri yalnızca hacim değil. Her senaryonun en az bir feature'da
ayırt edilebilir olduğu gösterilmeli. Gösterilemeyen senaryo ya güçlendirilir ya tezden
çıkarılır.

Skor = |saldırı ortalaması − normal ortalaması| / (normalin standart sapması + ε). Yani normalin doğal değişkenliği
cinsinden kaç "normal sapması" ayrıştığı. Ölçekten bağımsız olduğu için farklı
feature'lar karşılaştırılabilir. Eşik `vd.DISCRIMINATOR_THRESHOLD`.

In [ ]:
print(f"ayrılabilirlik eşiği: {vd.DISCRIMINATOR_THRESHOLD} normal σ")
fig, scores, scenarios = charts.separability_heatmap(df)
plt.show()

In [ ]:
# Aynı tablonun okunur hâli + "her scenario ayırt edilebilir mi?" sorusunun cevabı.
rows = []
for s in scenarios:
    best = vd.best_discriminator(df, s)
    rows.append({
        "scenario": s,
        "windows": int((df.label == s).sum()),
        "en iyi feature": best["feature"] if best else "—",
        "skor (normal σ)": round(best["score"], 1) if best else None,
        "saldırı ort.": round(best["attack_mean"], 1) if best else None,
        "normal ort.": round(best["normal_mean"], 1) if best else None,
        "dejenere": "evet" if best and best["degenerate"] else ("—" if not best else "hayır"),
        "status": style.status_text(best is not None, "separable", "NOT SEPARABLE"),
    })
best_by_scenario = pd.DataFrame(rows)
best_by_scenario

In [ ]:
# Degenerate feature ile ayrışan scenario'lar: ayırma "bedava", öğrenilecek bir şey yok.
degenerate = sorted({x["feature"] for s in scenarios
                     if (x := vd.best_discriminator(df, s)) and x["degenerate"]})
shared = {}
for s in scenarios:
    x = vd.best_discriminator(df, s)
    if x:
        shared.setdefault(x["feature"], []).append(s)

if degenerate:
    print("Şu feature'lar `normal` sınıfında HİÇ değişmiyor (std=0):")
    for c in degenerate:
        print(f"    {c} — degenerate: ayrışma 'bedava', öğrenilebilir sinyal değil")
for c, ls in shared.items():
    if len(ls) > 1:
        print(f"`{c}` tek başına {len(ls)} scenario'yu ayırıyor: {', '.join(ls)}")
        print("     → tek bir feature birden çok scenario'yu ayırıyorsa, model scenario'yu")
        print("       değil o feature'ın varlığını öğreniyor olabilir.")
if not degenerate and not any(len(v) > 1 for v in shared.values()):
    print("Degenerate feature'a dayanan ya da tek feature'la çoklu scenario ayıran tespit yok.")

## 6 · Feature dağılımları: normal ve saldırı

**Neden:** heatmap ortalamaların ayrışmasını gösteriyor; bu bölüm dağılımın tamamını
gösteriyor. Amaç, ölü feature'ları (iki sınıfta da hep sıfır) ve normalin değişkenliğini
gözle doğrulamak.

saldırı tarafı bütün senaryolar birleşik gösteriliyor; senaryo bazında ayrışma 5.
bölümde. Her feature kendi ekseninde çiziliyor, çünkü ölçekler kıyaslanamaz.

> **Nasıl okumalı:** normal kutusu bir hairline'a çöküyorsa, o feature normal sınıfında
> neredeyse hiç değişmiyor demektir; interquartile range sıfır. Başlığında "boş" yazan
> paneller iki sınıfta da tamamen sıfır (`is_degenerate`).

In [ ]:
import math

attack = df[df.label != "normal"]
normal = df[df.label == "normal"]
ncol = 4
nrow = math.ceil(len(ALL_COLS) / ncol)
fig, axes = plt.subplots(nrow, ncol, figsize=(11.4, 1.85 * nrow), sharey=False)

for ax, col in zip(axes.ravel(), ALL_COLS):
    data = [pd.to_numeric(normal[col], errors="coerce").dropna(),
            pd.to_numeric(attack[col], errors="coerce").dropna()]
    bp = ax.boxplot(data, orientation="horizontal", widths=0.5, patch_artist=True,
                    whis=1.5, positions=[1, 0])           # normal üstte, attack altta
    for box, color in zip(bp["boxes"], [style.S1, style.S2]):
        box.set(facecolor=color, edgecolor=style.SURFACE, linewidth=2, alpha=0.8)
    for median in bp["medians"]:
        median.set(color=style.INK, linewidth=1.5)        # median = kilit işaret
    for group in ("whiskers", "caps"):
        for artifact in bp[group]:
            artifact.set(color=style.AXIS, linewidth=0.8)
    for flier in bp["fliers"]:
        flier.set(marker="o", markersize=2, markerfacecolor=style.MUTED,
                  markeredgecolor="none", alpha=0.7)

    empty = vd.is_degenerate(df, col)
    ax.set_title(col + ("  —  boş" if empty else ""), fontsize=8.5,
                 color=(style.INK_2 if empty else style.INK))
    ax.set_yticks([1, 0])
    ax.set_yticklabels(["normal", "saldırı"])
    ax.tick_params(axis="y", length=0)
    ax.xaxis.grid(True)
    style.strip_frame(ax, ("top", "right", "left", "bottom"))
    if max(data[0].max() if len(data[0]) else 0, data[1].max() if len(data[1]) else 0) <= 0:
        ax.set_xlim(-0.5, 1)
        ax.text(0.5, 0.5, "her iki sınıfta da 0", transform=ax.transAxes,
                ha="center", va="center", color=style.MUTED, fontsize=8)

for ax in axes.ravel()[len(ALL_COLS):]:
    ax.axis("off")

fig.legend(handles=[Line2D([0], [0], marker="s", linestyle="none", markersize=8,
                           markerfacecolor=style.S1, markeredgecolor=style.SURFACE,
                           markeredgewidth=2, label="normal"),
                    Line2D([0], [0], marker="s", linestyle="none", markersize=8,
                           markerfacecolor=style.S2, markeredgecolor=style.SURFACE,
                           markeredgewidth=2, label="saldırı (6 senaryo birleşik)")],
           loc="upper right", ncol=2, bbox_to_anchor=(0.99, 1.045))
fig.suptitle("Feature dağılımları — her feature kendi ekseninde", x=0.02, ha="left",
             fontsize=10.5, color=style.INK)
plt.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## 7 · Window boyutu karşılaştırması

**Neden:** window kararı bir ödünleşim. Küçük window daha çok satır üretir ve kabul
criteria karşılanır, ama gürültüye duyarlıdır. Büyük window az satır üretir ve senaryo
başına window hedefinin altına düşer.

Bu bölüm ayrı çıktı dosyaları okuyor; `features.parquet` her window boyutu için aynı yola
yazıldığı için karşılaştırma ancak `FEATURES_OUT` verilerek yapılabilir. Dosya yoksa
bölüm ne çalıştırılacağını kendisi söylüyor.

In [ ]:
window_files = sorted((REPO / "src/data/features").glob("features-*s.parquet"))

if not window_files:
    print("Karşılaştırma için window başına ayrı çıktı dosyası yok.\n")
    print("Üretmek için (her biri ~1 koşu süresi kadar sürer):")
    print("  make features WINDOW=5s  FEATURES_OUT=data/features/features-5s.parquet")
    print("  make features WINDOW=10s FEATURES_OUT=data/features/features-10s.parquet")
    print("  make features WINDOW=30s FEATURES_OUT=data/features/features-30s.parquet")
else:
    rows = []
    for path in [FEATURES] + list(window_files):
        d = pd.read_parquet(path)
        sy = vd.summarize(d)
        rows.append({
            "file": path.name,
            "window": path.stem.replace("features", "").lstrip("-") or WINDOW,
            "total rows": sy["total_rows"],
            "normal": sy["normal_rows"],
            "min per scenario": sy["min_rows_per_scenario"],
            "scenario target": style.status_text(
                sy["min_rows_per_scenario"] >= vd.TARGETS["rows_per_scenario"]),
            "normalde kapsam": f"%{vd.normal_runtime_coverage(d) * 100:.1f}",
        })
    window_df = pd.DataFrame(rows).drop_duplicates(subset="file")

    fig, ax = plt.subplots(figsize=(7.6, 3.2))
    xi = range(len(window_df))
    ax.bar(xi, window_df["min per scenario"], width=0.5, color=style.S1, zorder=3)
    ax.axhline(vd.TARGETS["rows_per_scenario"], color=style.INK_2, lw=2, zorder=4)
    for i, v in zip(xi, window_df["min per scenario"]):
        ax.text(i, v + max(window_df["min per scenario"].max(), 1) * 0.03, str(v),
                ha="center", va="bottom", color=style.INK_2, fontsize=8)
    ax.set_xticks(list(xi))
    ax.set_xticklabels(window_df["window"])
    ax.set_xlabel("window boyutu")
    ax.set_ylabel("en az window'a sahip scenario")
    ax.set_title("Window boyutu vs scenario başına window sayısı")
    ax.yaxis.grid(True)
    style.strip_frame(ax, ("top", "right"))
    ax.annotate(f"target ≥{vd.TARGETS['rows_per_scenario']}",
                xy=(len(window_df) - 0.45, vd.TARGETS["rows_per_scenario"]),
                xytext=(0, 5), textcoords="offset points", color=style.INK_2,
                fontsize=8, ha="right")
    plt.tight_layout()
    plt.show()
    display(window_df)

## 8 · Archive kıyası: eski ve güncel veri seti

**Neden:** FAZ 3.5'in gerekçesi, eski veri setinin ölçülmüş kusurları. Bu bölüm o
kusurların canlı doğrulaması: aynı aynı fonksiyonlar eski veriye de uygulanıyor. Eski
set yeniden ölçüldüğünde rakamlar roadmap'te yazılanla tutmalı; tutmuyorsa roadmap'teki
iddiayı gözden geçirmek gerekiyor.

In [ ]:
sources = [("current", FEATURES)] + [
    (d.name.replace("arsiv-", "archive "), d / "features.parquet")
    for d in sorted((REPO / "src/data").glob("arsiv-*"))
    if (d / "features.parquet").is_file()]

rows = []
for name, path in sources:
    d = pd.read_parquet(path)
    sy = vd.summarize(d)
    undetectable = [s for s in sy["scenarios"] if vd.best_discriminator(d, s) is None]
    degenerate = sorted({x["feature"] for s in sy["scenarios"]
                         if (x := vd.best_discriminator(d, s)) and x["degenerate"]})
    rows.append({
        "dataset": name,
        "toplam satır": sy["total_rows"],
        "normal": sy["normal_rows"],
        "senaryo başına en az": sy["min_rows_per_scenario"],
        "normalde kapsam": round(vd.normal_runtime_coverage(d), 3),
        "ayrılamayan senaryo": len(undetectable),
        "dejenere feature kullanan": len(degenerate),
    })
comparison = pd.DataFrame(rows).drop_duplicates(subset="dataset")
comparison

In [ ]:
# Stat tile: iki veri setinin runtime kapsamı yan yana. İki çubukluk bir grafik
# yerine stat tile — sayı zaten hikâyenin tamamı.
target = vd.TARGETS["normal_runtime_coverage"]
fig, axes = plt.subplots(1, len(comparison), figsize=(4.0 * len(comparison), 1.9))
axes = np.atleast_1d(axes)
for ax, (_, r) in zip(axes, comparison.iterrows()):
    k = float(r["normalde kapsam"])
    ok = k >= target
    ax.axis("off")
    ax.text(0, 0.72, f"%{k * 100:.1f}", fontsize=32, color=style.INK, va="center", ha="left")
    ax.text(0, 0.38, f"{r['dataset']} · {r['toplam satır']} satır", fontsize=9,
            color=style.INK_2, va="center", ha="left")
    ax.text(0, 0.20, f"hedef ≥ %{target * 100:.0f}", fontsize=8, color=style.MUTED,
            va="center", ha="left")
    ax.text(0, 0.02, style.status_fig(ok), fontsize=10,
            color=(style.GOOD if ok else style.BAD), va="center", ha="left")
fig.suptitle("Normal sınıfında runtime kapsamı — eski ve yeni", x=0.02, ha="left",
             fontsize=10.5, color=style.INK)
plt.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

## 9 · Kabul kriterleri

`vd.check_targets` ile üretiliyor; `make validate` komutunun bastığı tablonun aynısı,
aynı fonksiyon ve aynı eşikler. Satır adları o fonksiyondan geldiği için Türkçe
kalıyor (bkz. baştaki sözlük notu). Kriter tutmazsa `make validate` sıfırdan farklı bir
kodla çıkıyor ve FAZ 4 bloke kalıyor.

In [ ]:
undetectable = [s for s in scenarios if vd.best_discriminator(df, s) is None]

raw = pd.DataFrame(vd.check_targets(summary, coverage, undetectable),
                   columns=["criterion", "actual", "target", "passed"])
all_pass = bool(raw["passed"].all())

criteria = raw.assign(status=[style.status_text(p) for p in raw["passed"]]) \
             .drop(columns=["passed"])
print("SONUÇ: " + ("kabul edilebilir — FAZ 4 açılabilir" if all_pass
                   else "FAZ 4'e hazır değil"))
if undetectable:
    print(f"  Ayırt edilemeyen scenario'lar: {', '.join(undetectable)}")
    print("  → güçlendir ya da tezden çıkar (roadmap FAZ 3.5 kabul kriterleri).")
criteria

In [ ]:
# Aynı tablo `make validate` ile de üretilir; ikisi çelişirse tek kaynak
# validate_dataset.py'dir. Dashboard yalnızca görünüm katmanıdır.
print("Doğrulamanın metin hâli:")
print("    make validate")